# 02. PDF 텍스트 레이어 검사

PyMuPDF로 원본 PDF의 내장 텍스트 레이어를 검사합니다. 이 노트북은 OCR 정확도를 측정하지 않으며, OCR 이전에 원본 PDF에서 신뢰할 만한 텍스트를 직접 추출할 수 있는지 구분합니다.

In [1]:
from collections import Counter
from pathlib import Path
import re

import fitz
import pandas as pd

PROJECT_ROOT = Path.cwd()
RAW_PDF_DIR = PROJECT_ROOT / 'data' / 'documents' / 'raw'
PDF_PATHS = sorted(RAW_PDF_DIR.glob('*/*.pdf'))

print(f'대상 PDF 수: {len(PDF_PATHS)}')
print(f'PyMuPDF 버전: {fitz.VersionBind}')

대상 PDF 수: 106
PyMuPDF 버전: 1.28.0


In [2]:
MIN_PAGE_CHARS = 20
MIN_DOCUMENT_CHARS = 200
USABLE_PAGE_RATIO = 0.80

def normalized_text(text: str) -> str:
    return re.sub(r'\s+', '', text)

def readable_ratio(text: str) -> float:
    compact = normalized_text(text)
    if not compact:
        return 0.0
    readable = sum(char.isalnum() or '\uac00' <= char <= '\ud7a3' or char in '%.,:/()[]-+~' for char in compact)
    return readable / len(compact)

def is_usable_page(text: str) -> bool:
    return len(normalized_text(text)) >= MIN_PAGE_CHARS and readable_ratio(text) >= 0.70

def classify_document(total_chars: int, usable_pages: int, page_count: int) -> str:
    usable_ratio = usable_pages / page_count if page_count else 0
    if total_chars >= MIN_DOCUMENT_CHARS and usable_ratio >= USABLE_PAGE_RATIO:
        return '텍스트 레이어 사용 가능'
    if total_chars > 0 and usable_pages > 0:
        return '부분 사용 가능'
    return 'OCR 필요'

def inspect_pdf(pdf_path: Path) -> dict:
    with fitz.open(pdf_path) as document:
        page_texts = [page.get_text('text') for page in document]
        page_char_counts = [len(normalized_text(text)) for text in page_texts]
        page_readable_ratios = [readable_ratio(text) for text in page_texts]

    page_count = len(page_char_counts)
    usable_pages = sum(is_usable_page(text) for text in page_texts)
    total_chars = sum(page_char_counts)
    return {
        'issuer': pdf_path.parent.name,
        'file_name': pdf_path.name,
        'page_count': page_count,
        'total_extracted_chars': total_chars,
        'usable_text_pages': usable_pages,
        'usable_page_ratio': usable_pages / page_count if page_count else 0,
        'min_page_chars': min(page_char_counts, default=0),
        'max_page_chars': max(page_char_counts, default=0),
        'avg_readable_ratio': sum(page_readable_ratios) / page_count if page_count else 0,
        'classification': classify_document(total_chars, usable_pages, page_count),
    }


In [3]:
results = []
errors = []

for pdf_path in PDF_PATHS:
    try:
        results.append(inspect_pdf(pdf_path))
    except Exception as error:
        errors.append({'issuer': pdf_path.parent.name, 'file_name': pdf_path.name, 'error': str(error)})

result_df = pd.DataFrame(results).sort_values(['classification', 'issuer', 'file_name'])
summary_df = (
    result_df.groupby('classification', sort=False)
    .agg(pdf_count=('file_name', 'count'), pages=('page_count', 'sum'), extracted_chars=('total_extracted_chars', 'sum'))
    .reindex(['텍스트 레이어 사용 가능', '부분 사용 가능', 'OCR 필요'], fill_value=0)
    .reset_index()
)

display(summary_df)
print(f'검사 완료: {len(result_df)}개, 오류: {len(errors)}개')
if errors:
    display(pd.DataFrame(errors))

,classification,pdf_count,pages,extracted_chars
0,텍스트 레이어 사용 가능,88,515,564938
1,부분 사용 가능,4,42,25935
2,OCR 필요,14,60,0


검사 완료: 106개, 오류: 0개


In [4]:
for classification in ['부분 사용 가능', 'OCR 필요']:
    print(f'\n[{classification}]')
    display(
        result_df.loc[result_df['classification'] == classification, [
            'issuer', 'file_name', 'page_count', 'total_extracted_chars',
            'usable_text_pages', 'usable_page_ratio', 'avg_readable_ratio', 'min_page_chars', 'max_page_chars'
        ]]
    )

print('\n[판정 기준]')
print(f'- 페이지 사용 가능: 공백 제거 후 {MIN_PAGE_CHARS}자 이상, 읽을 수 있는 문자 비율 70% 이상')
print(f'- 텍스트 레이어 사용 가능: 총 {MIN_DOCUMENT_CHARS}자 이상이고, 사용 가능 페이지 비율 {USABLE_PAGE_RATIO:.0%} 이상')
print('- 부분 사용 가능: 일부 페이지에서만 텍스트가 추출됨')
print('- OCR 필요: 사용 가능한 텍스트 페이지가 없음')


[부분 사용 가능]


,issuer,file_name,page_count,total_extracted_chars,usable_text_pages,usable_page_ratio,avg_readable_ratio,min_page_chars,max_page_chars
39,hyundai,Hyundai_The_Red_20260330.pdf,16,8141,12,0.75,0.911717,0,921
61,lotte,Lotte_Digiloca_Edu.pdf,8,5539,6,0.75,0.734811,0,1943
66,lotte,Lotte_Digiloca_Pet.pdf,8,5538,6,0.75,0.735072,0,1943
72,lotte,Lotte_LotteMart&MAXX.pdf,10,6717,7,0.70,0.774953,0,1981



[OCR 필요]


,issuer,file_name,page_count,total_extracted_chars,usable_text_pages,usable_page_ratio,avg_readable_ratio,min_page_chars,max_page_chars
6,BC,BC_KBank_SIMPLE.pdf,2,0,0,0.0,0.0,0,0
7,BC,BC_K_FRIST.pdf,2,0,0,0.0,0.0,0,0
41,ibk,IBK_DailyWith.pdf,6,0,0,0.0,0.0,0,0
42,ibk,IBK_Everyday_Joy_Credit.pdf,2,0,0,0.0,0.0,0,0
43,ibk,IBK_I-Anywhere_Green.pdf,6,0,0,0.0,0.0,0,0
44,ibk,IBK_IBK-Hybrid.pdf,4,0,0,0.0,0.0,0,0
45,ibk,IBK_IBKPoint(Credit).pdf,6,0,0,0.0,0.0,0,0
46,ibk,IBK_KPass(Credit).pdf,6,0,0,0.0,0.0,0,0
47,ibk,IBK_Point3.8(Credit).pdf,6,0,0,0.0,0.0,0,0
48,ibk,IBK_i-Mileage.pdf,6,0,0,0.0,0.0,0,0



[판정 기준]
- 페이지 사용 가능: 공백 제거 후 20자 이상, 읽을 수 있는 문자 비율 70% 이상
- 텍스트 레이어 사용 가능: 총 200자 이상이고, 사용 가능 페이지 비율 80% 이상
- 부분 사용 가능: 일부 페이지에서만 텍스트가 추출됨
- OCR 필요: 사용 가능한 텍스트 페이지가 없음


## 텍스트 품질 교차 검증

텍스트 레이어 사용 가능 PDF만 대상으로 PyMuPDF 결과를 기존 GPT Vision 전사본과 페이지 단위로 비교합니다. 자동 검사는 깨진 문자, 비정상 제어 문자, 숫자 토큰 불일치, 낮은 텍스트 겹침을 탐지합니다. 불일치는 PyMuPDF 오류로 단정하지 않고 원본 PDF 검토 대상으로 표시합니다.

In [5]:
import unicodedata

VISION_DIR = PROJECT_ROOT / 'data' / 'documents' / 'vision'
PAGE_PATTERN = re.compile(r'\[PAGE (\d+)\]\n(.*?)(?=\n\n-{20,}\n\n\[PAGE |\Z)', re.DOTALL)
TOKEN_PATTERN = re.compile(r'[가-힣A-Za-z]+|\d+(?:[,.]\d+)?%?')
NUMBER_PATTERN = re.compile(r'(?<!\d)\d{1,3}(?:,\d{3})*(?:\.\d+)?(?:원|%|개월|회|건|천원|만원)?')

def read_vision_pages(vision_path: Path) -> dict[int, str]:
    if not vision_path.exists():
        return {}
    content = vision_path.read_text(encoding='utf-8')
    return {int(number): text for number, text in PAGE_PATTERN.findall(content)}

def comparison_text(text: str) -> str:
    return re.sub(r'[^0-9a-z가-힣%]+', '', text.lower())

def token_set(text: str) -> set[str]:
    return set(TOKEN_PATTERN.findall(text.lower()))

def jaccard_score(left: set[str], right: set[str]) -> float:
    union = left | right
    return len(left & right) / len(union) if union else 1.0

def character_issues(text: str) -> tuple[list[str], list[str]]:
    critical = sorted({
        char for char in text
        if char == '\ufffd' or unicodedata.category(char) in {'Cn', 'Co'}
    })
    controls = sorted({
        char for char in text
        if unicodedata.category(char) == 'Cc' and not char.isspace()
    })
    return critical, controls

def validate_page(pdf_text: str, vision_text: str | None) -> dict:
    critical_chars, control_chars = character_issues(pdf_text)
    pdf_numbers = set(NUMBER_PATTERN.findall(pdf_text))
    if vision_text is None:
        return {
            'status': '원본 확인 필요',
            'reason': 'Vision 페이지 결과 없음',
            'token_jaccard': None,
            'number_recall': None,
            'critical_chars': ''.join(critical_chars),
            'control_chars': ''.join(control_chars),
        }

    token_jaccard = jaccard_score(token_set(pdf_text), token_set(vision_text))
    vision_numbers = set(NUMBER_PATTERN.findall(vision_text))
    number_recall = len(pdf_numbers & vision_numbers) / len(pdf_numbers) if pdf_numbers else 1.0

    if not normalized_text(pdf_text):
        status, reason = 'OCR 보완 필요', 'PyMuPDF 추출 텍스트가 없음'
    elif critical_chars:
        status, reason = '깨짐 확인', '정의되지 않았거나 사설 영역의 문자 존재'
    elif control_chars:
        status, reason = '정제 필요', '보이지 않는 제어 문자 존재'
    elif token_jaccard < 0.45:
        status, reason = '원본 확인 필요', 'Vision 결과와 텍스트 토큰 겹침이 낮음'
    elif number_recall < 0.60:
        status, reason = '원본 확인 필요', 'PyMuPDF 숫자 토큰의 40% 초과가 Vision 결과에 없음'
    else:
        status, reason = '자동 이상 징후 없음', ''

    return {
        'status': status,
        'reason': reason,
        'token_jaccard': token_jaccard,
        'number_recall': number_recall,
        'critical_chars': ''.join(critical_chars),
        'control_chars': ''.join(control_chars),
    }

In [6]:
text_layer_pdfs = result_df.loc[result_df['classification'] == '텍스트 레이어 사용 가능', ['issuer', 'file_name']]
validation_rows = []

for record in text_layer_pdfs.itertuples(index=False):
    pdf_path = RAW_PDF_DIR / record.issuer / record.file_name
    vision_path = VISION_DIR / record.issuer / f'{pdf_path.stem}.txt'
    vision_pages = read_vision_pages(vision_path)

    with fitz.open(pdf_path) as document:
        for page_number, page in enumerate(document, start=1):
            pdf_text = page.get_text('text')
            check = validate_page(pdf_text, vision_pages.get(page_number))
            validation_rows.append({
                'issuer': record.issuer,
                'file_name': record.file_name,
                'page_number': page_number,
                'pdf_chars': len(normalized_text(pdf_text)),
                **check,
            })

validation_df = pd.DataFrame(validation_rows)
validation_summary = (
    validation_df.groupby('status', sort=False)
    .agg(page_count=('page_number', 'count'), pdf_count=('file_name', 'nunique'))
    .reindex(['자동 이상 징후 없음', '정제 필요', '원본 확인 필요', '깨짐 확인', 'OCR 보완 필요'], fill_value=0)
    .reset_index()
)

display(validation_summary)
print(f'검증 대상: PDF {len(text_layer_pdfs)}개, 페이지 {len(validation_df)}개')

,status,page_count,pdf_count
0,자동 이상 징후 없음,217,51
1,정제 필요,225,49
2,원본 확인 필요,39,19
3,깨짐 확인,11,8
4,OCR 보완 필요,23,16


검증 대상: PDF 88개, 페이지 515개


In [7]:
review_df = validation_df.loc[validation_df['status'] != '자동 이상 징후 없음'].sort_values(['status', 'issuer', 'file_name', 'page_number'])

if review_df.empty:
    print('자동 검사에서 깨짐 또는 교차 검증 불일치 페이지가 발견되지 않았습니다.')
else:
    display(review_df[[
        'status', 'reason', 'issuer', 'file_name', 'page_number', 'pdf_chars',
        'token_jaccard', 'number_recall', 'critical_chars', 'control_chars'
    ]])

print('\n해석: 자동 이상 징후 없음은 원본과의 완전 일치를 의미하지 않습니다.')
print('낮은 토큰 겹침과 숫자 불일치는 PyMuPDF 또는 Vision 중 어느 쪽의 오류인지 원본 PDF로 확인해야 합니다.')

,status,reason,issuer,file_name,page_number,pdf_chars,token_jaccard,number_recall,critical_chars,control_chars
105,OCR 보완 필요,PyMuPDF 추출 텍스트가 없음,hyundai,Hyundai_D_250827.pdf,1,0,0.000000,1.000000,,
106,OCR 보완 필요,PyMuPDF 추출 텍스트가 없음,hyundai,Hyundai_D_250827.pdf,2,0,0.000000,1.000000,,
115,OCR 보완 필요,PyMuPDF 추출 텍스트가 없음,hyundai,Hyundai_H_250827.pdf,1,0,0.000000,1.000000,,
126,OCR 보완 필요,PyMuPDF 추출 텍스트가 없음,hyundai,Hyundai_M.pdf,1,0,0.000000,1.000000,,
139,OCR 보완 필요,PyMuPDF 추출 텍스트가 없음,hyundai,Hyundai_O_250827.pdf,1,0,0.000000,1.000000,,
...,...,...,...,...,...,...,...,...,...,...
510,정제 필요,보이지 않는 제어 문자 존재,woori,Woori_Classic_TEN.pdf,2,2868,0.888372,1.000000,,
511,정제 필요,보이지 않는 제어 문자 존재,woori,Woori_D4_Classic_II.pdf,1,1759,0.916216,0.956522,,
512,정제 필요,보이지 않는 제어 문자 존재,woori,Woori_D4_Classic_II.pdf,2,2828,0.945833,1.000000,,
513,정제 필요,보이지 않는 제어 문자 존재,woori,Woori_D4_Classic_II_.pdf,1,2879,0.907609,1.000000,,



해석: 자동 이상 징후 없음은 원본과의 완전 일치를 의미하지 않습니다.
낮은 토큰 겹침과 숫자 불일치는 PyMuPDF 또는 Vision 중 어느 쪽의 오류인지 원본 PDF로 확인해야 합니다.


## PyMuPDF 원문 텍스트 저장

텍스트 레이어 사용 가능 PDF의 PyMuPDF 추출 결과를 엔진별 원본으로 보관합니다. 기존 Vision OCR 결과를 덮어쓰지 않습니다.

In [8]:
PYMUPDF_OUTPUT_DIR = PROJECT_ROOT / 'notebooks' / 'data' / 'documents' / 'pymupdf'
saved_paths = []

for record in text_layer_pdfs.itertuples(index=False):
    pdf_path = RAW_PDF_DIR / record.issuer / record.file_name
    output_path = PYMUPDF_OUTPUT_DIR / record.issuer / f'{pdf_path.stem}.txt'

    with fitz.open(pdf_path) as document:
        pages = [f'[PAGE {page_number}]\n{page.get_text("text").rstrip()}' for page_number, page in enumerate(document, start=1)]

    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text('\n\n' + ('\n\n' + ('-' * 80) + '\n\n').join(pages) + '\n', encoding='utf-8')
    saved_paths.append(output_path)

print(f'저장 완료: {len(saved_paths)}개')
print(f'저장 경로: {PYMUPDF_OUTPUT_DIR}')

저장 완료: 88개
저장 경로: notebooks/data/02_pdf_text_layer_check/documents/pymupdf


## PyMuPDF와 Vision OCR 표본 비교

PyMuPDF에서 빈 페이지, 제어 문자, 사설 영역 문자가 없는 문서를 표본으로 선택해 기존 Vision OCR 전사 결과와 비교합니다. 이 비교는 두 결과의 일치 정도를 보는 것이며, 어느 결과가 원본과 완전히 동일하다는 뜻은 아닙니다.

In [9]:
COMPARISON_SAMPLES = [
    ('BC', 'BC_Baro_Clear_Plus.pdf'),
    ('hana', 'Hana_Everyones_Shinsegae.pdf'),
    ('kookmin', 'Kookmin_Coupang_Wow_20250702.pdf'),
    ('lotte', 'Lotte_DIGILOCA_SKYPASS.pdf'),
    ('shinhan', 'Shinhan_Air_One_20241224.pdf'),
]

def read_pymupdf_pages(text_path: Path) -> dict[int, str]:
    content = text_path.read_text(encoding='utf-8')
    return {int(number): text for number, text in PAGE_PATTERN.findall(content)}

comparison_rows = []
page_pairs = {}

for issuer, pdf_name in COMPARISON_SAMPLES:
    stem = Path(pdf_name).stem
    pymupdf_pages = read_pymupdf_pages(PYMUPDF_OUTPUT_DIR / issuer / f'{stem}.txt')
    vision_pages = read_vision_pages(VISION_DIR / issuer / f'{stem}.txt')
    page_pairs[(issuer, pdf_name)] = (pymupdf_pages, vision_pages)

    for page_number, pymupdf_text in pymupdf_pages.items():
        vision_text = vision_pages.get(page_number, '')
        pymupdf_tokens = token_set(pymupdf_text)
        vision_tokens = token_set(vision_text)
        pymupdf_numbers = set(NUMBER_PATTERN.findall(pymupdf_text))
        vision_numbers = set(NUMBER_PATTERN.findall(vision_text))
        comparison_rows.append({
            'issuer': issuer,
            'file_name': pdf_name,
            'page_number': page_number,
            'pymupdf_chars': len(normalized_text(pymupdf_text)),
            'vision_chars': len(normalized_text(vision_text)),
            'token_jaccard': jaccard_score(pymupdf_tokens, vision_tokens),
            'numeric_token_overlap': jaccard_score(pymupdf_numbers, vision_numbers),
            'vision_page_found': bool(vision_text),
        })

comparison_df = pd.DataFrame(comparison_rows)
comparison_summary = (
    comparison_df.groupby(['issuer', 'file_name'], as_index=False)
    .agg(
        pages=('page_number', 'count'),
        pymupdf_chars=('pymupdf_chars', 'sum'),
        vision_chars=('vision_chars', 'sum'),
        mean_token_jaccard=('token_jaccard', 'mean'),
        mean_numeric_overlap=('numeric_token_overlap', 'mean'),
        vision_pages_found=('vision_page_found', 'sum'),
    )
)
comparison_summary['char_count_ratio_pymupdf_to_vision'] = (
    comparison_summary['pymupdf_chars'] / comparison_summary['vision_chars']
)
display(comparison_summary.round(4))
display(comparison_df.sort_values(['issuer', 'page_number']).round(4))

,issuer,file_name,pages,pymupdf_chars,vision_chars,mean_token_jaccard,mean_numeric_overlap,vision_pages_found,char_count_ratio_pymupdf_to_vision
0,BC,BC_Baro_Clear_Plus.pdf,2,4352,4243,0.8839,0.9800,2,1.0257
1,hana,Hana_Everyones_Shinsegae.pdf,1,3599,3629,0.8081,0.8824,1,0.9917
2,kookmin,Kookmin_Coupang_Wow_20250702.pdf,3,3208,3257,0.1741,0.9744,3,0.9850
3,lotte,Lotte_DIGILOCA_SKYPASS.pdf,7,5827,5986,0.8924,0.9951,7,0.9734
4,shinhan,Shinhan_Air_One_20241224.pdf,2,4828,4881,0.8694,1.0000,2,0.9891


,issuer,file_name,page_number,pymupdf_chars,vision_chars,token_jaccard,numeric_token_overlap,vision_page_found
0,BC,BC_Baro_Clear_Plus.pdf,1,2307,2192,0.8133,0.9600,True
1,BC,BC_Baro_Clear_Plus.pdf,2,2045,2051,0.9544,1.0000,True
2,hana,Hana_Everyones_Shinsegae.pdf,1,3599,3629,0.8081,0.8824,True
3,kookmin,Kookmin_Coupang_Wow_20250702.pdf,1,1115,1114,0.2391,1.0000,True
4,kookmin,Kookmin_Coupang_Wow_20250702.pdf,2,1206,1203,0.1403,1.0000,True
5,kookmin,Kookmin_Coupang_Wow_20250702.pdf,3,887,940,0.1429,0.9231,True
6,lotte,Lotte_DIGILOCA_SKYPASS.pdf,1,1410,1437,0.9892,1.0000,True
7,lotte,Lotte_DIGILOCA_SKYPASS.pdf,2,15,27,0.4000,1.0000,True
8,lotte,Lotte_DIGILOCA_SKYPASS.pdf,3,646,713,1.0000,1.0000,True
9,lotte,Lotte_DIGILOCA_SKYPASS.pdf,4,609,610,0.9645,1.0000,True


In [10]:
PREVIEW_CHARS = 700

for issuer, pdf_name in COMPARISON_SAMPLES:
    pymupdf_pages, vision_pages = page_pairs[(issuer, pdf_name)]
    page_number = min(pymupdf_pages)
    print('=' * 100)
    print(f'{issuer}/{pdf_name} - PAGE {page_number}')
    print('\n[PyMuPDF]')
    print(pymupdf_pages[page_number][:PREVIEW_CHARS])
    print('\n[Vision OCR]')
    print(vision_pages.get(page_number, '')[:PREVIEW_CHARS])
    print()

BC/BC_Baro_Clear_Plus.pdf - PAGE 1

[PyMuPDF]
부가서비스 안내
카드 이용 시 제공되는 포인트 및  할인혜택 등의 부가서비스는 카드 신규 출시(2022년
4월 1일) 이후 3년 이상 축소·폐지 없이 유지됩니다.
상기에도 불구하고, 다음과 같은 사유가 발생한 경우 카드사는 부가서비스를 변경할 수 있습니다. 
① 카드사의 휴업·파산·경영상의 위기 등에 따른 불가피한 경우
    ①의2. 제휴업체의 휴업·파산·경영상의 위기로 인해 불가피하게 부가서비스를 축소·변경
    하는 경우로서 다른 제휴업체를 통해 동종의 유사한 부가서비스 제공이 불가한 경우  
② 제휴업체가 카드사의 의사에 반하여 해당 부가서비스를 축소하거나 변경 시, 당초
     부가서비스에 상응하는 다른 부가서비스를 제공하는 경우 
③ 부가서비스를 3년 이상 제공한 상태에서 해당 부가서비스로 인해 상품의 수익성이      
     현저히 낮아진 경우
카드사가 부가서비스를 변경하는 경우 변경 사유, 변경 내용 등을 사유발생 즉시 아래 고지
방법 중 2가지 이상의 방법으로 고지하여 드립니다. 특히 부가서비스를  3년 이상 제공한 
상태에서 해당 부가서비스로 인해 상품의 수익성이 현저히 낮아져 부가서비스를 변경하는 
경우에는 6개월 전부터 아래 고지방법 중 2가지 이상의 방법으로 매월 고지하여 드립니다.
【고지 방법】
서면 교부, 우편 또는 전자우편, 전화 또는 팩스, 휴대폰 메시지 또는 이에 준하는 전자적 의사표시
·
·


[Vision OCR]
부가서비스 안내
· 카드 이용 시 제공되는 포인트 및 할인혜택 등의 부가서비스는 카드 신규 출시(2022년 4월 1일) 이후 3년 이상 축소·폐지 없이 유지됩니다.
· 상기에도 불구하고 다음과 같은 사유가 발생한 경우 카드는 부가서비스를 변경할 수 있습니다.
  ① 카드사의 휴업·파산·경영상의 위기 등에 따른 불가피한 경우
    ①의2. 제휴업체의 휴업·파산·경영상의 위기 또는 은행 파산 등에 따라 제휴업체와의 제휴를 통해

## 원본 PDF 기반 OCR 정량 평가 (10페이지 표본)

카드 혜택·연회비·조건 등 실제 정보가 있는 10개 페이지를 선정했습니다. 각 페이지에는 전체 텍스트 정답과 핵심 필드 정답을 모두 작성했습니다. 전체 텍스트 정답은 기존 PyMuPDF/Vision 결과가 아니라 렌더링한 원본 PDF 페이지를 보고 전사한 값입니다. 읽기 순서와 공백·글머리표의 표현 방식은 엔진마다 달라질 수 있으므로, CER/WER 계산 전 공백·제어문자·글머리표만 정규화합니다.

이 표본은 초기 진단용입니다. 10페이지 결과만으로 전체 PDF 또는 엔진 전체 성능을 일반화할 수 없습니다.

In [11]:
import json

# 원본 PDF 렌더링을 눈으로 확인해 작성한 페이지 전체 정답 전사본입니다.
# 표·로고의 시각적 배치 자체는 평가하지 않고, 사람이 읽을 수 있는 텍스트만 기록합니다.
FULL_TEXT_GOLD_SAMPLES = [
    {
        'issuer': 'samsung',
        'file_name': 'Samsung_5_V4.pdf',
        'page_number': 1,
        'gold_text': '''FOR EARTH FOR US 삼성카드는 저탄소 인증종이와 콩기름잉크를 사용하여 환경 사랑을 실천합니다.
609
삼성카드 5 V4
※ 상환 능력에 비해 신용카드 사용액이 과도할 경우 귀하의 개인신용평점이 하락할 수 있습니다.
※ 개인신용평점 하락 시 금융거래 관련된 불이익이 발생할 수 있습니다.
※ 일정 기간 원리금을 연체할 경우, 모든 원리금을 변제할 의무가 발생할 수 있습니다.
※ 금융상품 이용 전 상품설명서, 홈페이지, 약관을 통해 이용조건을 확인해 주시기 바랍니다.
※ 필요 이상으로 신용카드를 발급 및 이용하실 경우 개인신용평점, 이용한도 등에 영향을 미칠 수 있습니다.
※ 금융소비자는 해당 상품 또는 서비스에 대하여 설명을 받을 권리가 있습니다.
※ 신용카드 발급이 부적정한 경우(개인신용평점 낮음 등) 카드 발급이 제한될 수 있습니다.
※ 카드 이용대금과 이에 수반되는 모든 수수료를 지정된 대금 결제일에 상환합니다.
※ 여신금융협회 심의필 제 2020-C1a-00570호(2020.01.22)
본 상품안내장(상품약관)은 계약서류의 일부로서, 법령 및 내부통제기준에 따른 절차를 거쳐 제공됩니다.
대표전화 1588-8700
분실신고 1588-8900
www.samsungcard.com
2023.4
Samsung Card''',
    },
    {
        'issuer': 'lotte',
        'file_name': 'Lotte_LOCA_LIKIT_Play.pdf',
        'page_number': 3,
        'gold_text': '''LOCA LIKIT Play 혜택 안내
★ 주유
SK에너지, S-OIL, GS칼텍스, 현대오일뱅크
60% 결제일 할인
LPG 충전소 이용금액은 혜택이 제공되지 않습니다.
★ 영화
롯데시네마, CGV, 메가박스 60% 결제일 할인
현장 결제, 공식 홈페이지 및 앱을 통한 예매 시 할인 혜택이 제공되며, 기프티콘, 상품권 구매 등 영화 예매 이외의 건은 혜택이 제공되지 않습니다.
★ 스트리밍
넷플릭스, 왓챠, 유튜브, 멜론, 지니
60% 결제일 할인
공식 홈페이지 및 앱을 통해 직접 신청한 정기결제(자동납부) 이용 건만 혜택이 제공되며, 1회성 일반구매 결제 및 통신요금 내 합산 청구, 앱마켓(구글플레이·앱스토어 등) 결제와 같이 가맹점 구분이 불가한 경우에는 혜택이 제공되지 않습니다.
★ 멤버십
쿠팡 로켓와우, 네이버플러스 멤버십
60% 결제일 할인
★ 할인 공통기준
지난달 1일~말일까지 LOCA LIKIT Play 카드로 40만원 이상 이용 시 혜택이 제공됩니다.
할인한도는 모든 혜택 통합 월 1만 3천원 한도 내에서 제공합니다.
롯데카드 가맹점 및 업종 분류 기준으로 혜택을 제공합니다.
할인 적용 제외 대상
모든 무이자할부 이용금액, 포인트 충전, 기프트·선불카드 충전 및 구매, 상품권(모바일 상품권 포함) 구매''',
    },
    {
        'issuer': 'shinhan',
        'file_name': 'Shinhan_Discount_Plan+_20250509.pdf',
        'page_number': 1,
        'gold_text': '''고객센터 1544-7000
카드신청 1661-8599
오토다이렉트 센터 1688-7474
단기/장기 카드대출 1544-0303
MF대출서비스 1661-2032
할부금융 고객센터 1544-7100
신한카드 Discount Plan+
계획한 모든 일상에 따라오는 할인
※ 연체이자율은 ‘회원별, 이용상품별 약정금리+최대 연 3%, 법정 최고금리(연 20%) 이내’에서 적용됩니다.
단, 연체 발생 시점에 약정금리가 없는 경우 약정금리는 아래와 같이 적용함
- 일시불 거래 연체 시 : 거래 발생 시점의 최소기간(2개월) 유이자 할부 금리
- 무이자 할부 거래 연체 시 : 거래 발생 시점의 동일한 할부 계약기간의 유이자 할부 금리
- 그 외의 경우 : 상법상 상사법정이율과 상호금융 가계자금대출금리* 중 높은 금리 적용
* 한국은행에서 매월 발표하는 가장 최근의 비은행 금융기관 가중평균대출금리(신규대출 기준)
※ 상환능력에 비해 신용카드 사용액이 과도한 경우, 귀하의 개인신용평점이 하락할 수 있습니다.
※ 개인신용평점 하락시 금융거래와 관련된 불이익이 발생할 수 있습니다.
※ 일정기간 신용카드 이용대금을 연체할 경우, 결제일이 도래하지 않은 모든 신용카드 이용대금을 변제할 의무가 발생할 수 있습니다.
※ 여신금융협회 심의필 제2025-C1a-05552호(2025.04.29)''',
    },
    {
        'issuer': 'NH',
        'file_name': 'NH_AllWonderful.pdf',
        'page_number': 1,
        'gold_text': '''한 장의 카드로 두가지 혜택 경험!
NH올원더풀
모든 순간, 원더풀하게 채워지다
발급기준
발급대상 개인(신용)
교통구분 교통(후불), 비교통
※ 가족카드 발급 가능
연회비 안내
구분 브랜드 본인카드 가족카드
총연회비 기본연회비 제휴연회비 총연회비 기본연회비 제휴연회비
국내전용 Local 28,000원 6,000원 22,000원 22,000원 0원 22,000원
국내외겸용 Mastercard UnionPay 30,000원 6,000원 24,000원 24,000원 0원 24,000원
카드연회비(기본연회비+제휴연회비)는 보유 카드별로 청구됩니다.
유효기간이 도래하기 전에 카드를 해지하는 경우 연회비 반환금액은 고객님이 NH농협카드와 계약을 해지한 날부터 일할 계산하여 산정됩니다.
- 단, 카드의 발행·배송 등 카드 발급(신규 가입년도에 한함)에 소요된 비용, 카드 이용 시 제공되는 추가적인 혜택 등 부가서비스 제공에 소요된 비용은 반환금액에서 제외됩니다.
연회비 반환은 계약을 해지한 날부터 10영업일 이내 반환하여 드립니다.
- 단, 부가서비스 제공내역 확인에 시간이 소요되는 등의 불가피한 사유 시에는 계약을 해지한 날부터 3개월 이내 반환할 수 있습니다.
할인 PACK과 적립 PACK 중 원하시는 서비스로 직접 선택하실 수 있습니다. (월 1회 변경가능)
서비스팩 변경 신청은 NH농협카드 홈페이지(card.nonghyup.com), NH pay(NH페이)앱, 카드고객상담센터, 영업점을 통해 가능하며, 변경 신청 접수 후 익월 1일부터 변경된 선택 서비스가 적용됩니다.''',
    },
    {
        'issuer': 'hyundai',
        'file_name': 'Hyundai_T_20260319.pdf',
        'page_number': 5,
        'gold_text': '''우대 서비스
해외 온·오프라인 가맹점 국제브랜드 수수료 및 해외서비스 수수료 100% 청구 할인
청구 할인 대상 수수료율
- 국제브랜드 수수료(Visa) : 1.1%
- 해외서비스 수수료 : 0.2%
실적 조건 및 할인 한도 없음
수수료 할인은 국내외겸용 카드에 한해 적용
해외 온라인 가맹점 이용 시, 국내 가맹점번호로 승인 처리되는 일부 결제 건(간편결제, PG결제(결제대행) 등)은 제외
모든 가맹점은 현대카드 가맹점 등록 및 업종 분류 기준
해외 이용 관련 자세한 내용은 카드 이용 유의사항 참고
국제브랜드 수수료 및 해외서비스 수수료는 해외 이용 시 전액 부과된 후 카드 이용 대금 청구일에 100% 할인 적용된 금액으로 청구
메탈 플레이트 제공
본인 회원 신청 시 제공되며, 발급 수수료 7만원 별도 부과(재발급 및 갱신 시 동일 적용)
국내외겸용(Visa Platinum) 카드 신청 시 발급 가능
일부 ATM 기기에서 이용 불가
메탈 플레이트 특성상 수작업으로 제작되어 신청일로부터 발급까지 평균 3주 내외 소요(플레이트 수급 상황에 따라 소요 기간은 달라질 수 있음)
본인 카드 기본 플레이트와 유효 기간이 동일하며, 기본 플레이트 해지 시 메탈 플레이트도 동시 해지''',
    },
    {
        'issuer': 'BC',
        'file_name': 'BC_Business_Sky.pdf',
        'page_number': 1,
        'gold_text': '''Business Sky카드
공통 서비스 카드
Business Sky카드
필요 이상으로 신용카드를 발급 받으신 경우 회원님의 신용등급 또는 개인신용평점이나 이용 한도에 영향을 미칠 수 있습니다.
연체이자율 : 회원별, 이용상품별 약정이율 + 최대 3%, (단, 법정 최고금리(20%)이내)
단, 연체 발생 시점에 약정금리가 없는 경우는 아래와 같이 적용합니다.
- 일시불 거래 연체 시 : 거래발생 시점의 최소기간(2개월) 유이자 할부금리
- 무이자 할부 거래 연체 시 : 거래 발생 시점의 동일한 할부 계약기간의 유이자 할부 금리
- 그 외의 경우 : 약정금리는 상법상 상사법정이율과 상호금융 가계자금대출금리*중 높은 금리적용
*한국은행에서 매월 발표하는 가장 최근의 비은행 금융기관 가중평균대출금리(신규대출 기준)
상환능력에 비해 신용카드 사용액이 과도할 경우, 귀하의 개인신용평점이 하락할 수 있습니다.
개인신용평점 하락 시 금융거래와 관련된 불이익이 발생할 수 있습니다.
일정기간 원리금을 연체할 경우, 모든 원리금을 변제할 의무가 발생할 수 있습니다.
신용카드 발급이 부적정한 경우(연체금 보유, 신용점수 낮음 등) 카드발급이 제한될 수 있습니다.
카드 이용대금과 이에 수반되는 모든 수수료를 지정된 대금 결제일에 상환합니다.
금융소비자는 금소법 제19조 제1항에 따라 해당상품 또는 서비스에 대하여 설명을 받을 권리가 있으며, 그 설명을 듣고 내용을 충분히 이해한 후 거래하시기 바랍니다.
준법감시인 2021-1725호(기준일: 2021.11.05)''',
    },
    {
        'issuer': 'hana',
        'file_name': 'Hana_WonderCard2.0.pdf',
        'page_number': 12,
        'gold_text': '''2. 원더 서비스
3) 온가족 플러스
온가족 플러스
온가족 플러스 신청 시, 본인가드와 가족카드별 실적에 따라 월 혜택과 연 혜택을 제공합니다.
월 혜택
혜택 가족카드당 1만원 캐시백(월 1회)
제공기준 실적 조건
- 1차년도 : 본인 및 가족카드별 지난달 40만원 이상 이용
- 2차년도 이후 : ①, ② 조건 충족 시
① 본인 및 가족카드별 지난달 40만원 이상 이용
② 본인 및 가족카드별 직전 6개월 연속 1원 이상 이용
제공 방식 실적 조건 충족 다음달 10영업일 이내 제공
* 기본서비스 서비스 방식이 적립인 경우, 1만 하나머니 적립으로 제공
연 혜택
혜택 차년도 본인가드 연회비 지원 1만원 캐시백(연 1회)
제공기준 실적 조건 본인 및 가족카드 직전 6개월 연속 1원 이상 이용
제공 방식 차년도 실적 조건 충족 다음달 10영업일 이내 제공
* 2차년도 부터 서비스 제공
온가족 플러스 최초 신청월을 서비스 시작월로 하며, 온가족 플러스 신청 후 가족카드 발급 시에는 최초 가족카드 발급 완료월을 서비스 시작월로 월 혜택과 연 혜택을 제공합니다.
- 1차년도 : 서비스 시작월 포함 12개월
- 2차년도 이후 : 전년도 서비스 제공기간 종료 후 12개월
원더카드 2.0 본인가드 해지 후 다시 본인가드를 신규로 발급하는 경우에는 서비스 시작월을 재산정 합니다.
온가족 플러스 최초 서비스 실적은 서비스 시작월 전체 이용금액 기준으로 산정합니다.
온가족 플러스 월 혜택은 서비스 실적 조건 충족 시, 다음달 10영업일 이내 제공되며 가족카드 최대 4장까지 혜택 제공됩니다.''',
    },
    {
        'issuer': 'kookmin',
        'file_name': 'Kookmin_My_WE_SH_20250102.pdf',
        'page_number': 3,
        'gold_text': '''더욱 진심 서비스 (서비스팩 3개중 택1)
[먹는데 진심] 배달/커피 5% 할인
배달업(배달의민족, 요기요, 마켓컬리), 커피(커피/음료전문점 업종)
상품권(선물하기 등), 선불카드(선불전자지급수단 포함) 제외
배달 현장 결제건 제외
백화점/대형마트 등 일부 입점 매장 제외
[노는데 진심] 택시/커피 5%, 영화관 30% 할인
택시, 커피: 택시, 커피/음료전문점 업종
영화관: CGV, 롯데시네마, 메가박스
매점, 관람권(상품권) 구입, 예매대행사이트는 제외
[관리에 진심] 미용실, 스포츠, 온라인서점, 올리브영 5% 할인
미용실, 스포츠: 미용실, 종합스포츠센터, 골프(연습)장, 테니스장, 수영장, 요가, 볼링장, 스포츠용품점, 레저용품점 업종
온라인서점: 교보문고, YES24 공식 홈페이지(앱)
올리브영: 오프라인 매장 및 온라인 공식 쇼핑몰(앱)
더욱 진심 서비스 월 할인한도
먹는데 진심 배달, 커피 5% 5천원
노는데 진심 택시, 커피 5% 5천원
노는데 진심 영화관 30% 5천원
관리에 진심 미용실, 스포츠, 온라인서점, 올리브영 5% 1만원
전월 이용실적 40만원 이상 시 제공
영화관 할인은 연 4회(연 2만원) 이내 제공
최초 발급받은 My WE:SH 카드 사용등록일(KB Pay 등 간편결제 등록 포함)로부터 다음달 말일(실적유예기간)까지는 전월 이용실적 40만원 미만시에도 할인서비스 제공''',
    },
    {
        'issuer': 'samsung',
        'file_name': 'Samsung_iD_ALL.pdf',
        'page_number': 2,
        'gold_text': '''많이 쓰는 영역
5% 자동 맞춤 할인
백화점·할인점·슈퍼마켓 영역 중 월 이용금액이 가장 큰 1개 영역에 대해 5% 결제일할인
영역 할인 대상
백화점 신세계/롯데/현대/갤러리아 백화점, AK플라자
할인점 이마트, 이마트 트레이더스, 롯데마트, 홈플러스, 농협 하나로마트
슈퍼마켓 이마트 에브리데이, GS THE FRESH(구. GS 수퍼마켓), 롯데슈퍼, 홈플러스 익스프레스
전월 이용금액대별 월 할인한도
40만원 이상 5,000원
70만원 이상 10,000원
발급월+1개월까지는 전월 이용금액 40만원 미만 시에도 40만원 이상~70만원 미만 실적구간 혜택 제공(전월 이용금액 70만원 이상 시에는 해당 실적구간 혜택 제공)
국내외 가맹점 0.5% 할인 혜택과 중복 적용
결제일할인금액은 다음 달 15일 이후 결제대금에서 차감
3개 영역 월 합산 이용금액 1원 이상 시 제공(1개 영역 이상 이용 시 제공)
결제 취소건의 경우, 매출취소전표 접수월의 3개 영역 합산 이용금액 및 영역별 이용금액에 반영
오프라인 결제건에 한하며, 쇼핑 외 결제건(상품권, 주차장 등), 임대매장은 제외
가족카드에는 제공되지 않음''',
    },
    {
        'issuer': 'lotte',
        'file_name': 'Lotte_LOCA_LIKIT_Eat.pdf',
        'page_number': 3,
        'gold_text': '''LOCA LIKIT Eat 혜택 안내
음식점 음식점 60% 결제일 할인
음식점 업종으로 등록된 가맹점에서 혜택이 제공되며, 주점, 유흥업소, 커피전문점, 베이커리, 제과점 및 백화점·마트 등에 입점한 음식점 이용금액은 혜택이 제공되지 않습니다.
배달앱 배달의 민족, 쿠팡이츠, 요기요 60% 결제일 할인
공식 홈페이지 및 앱을 통한 결제 건에만 혜택이 제공되며, 배달원 또는 가맹점 직접 결제 건은 혜택이 제공되지 않습니다.
커피 스타벅스, 투썸플레이스, 할리스커피, 폴바셋 60% 결제일 할인
백화점, 할인점, 쇼핑몰 등에 입점한 매장에서는 혜택이 제공되지 않습니다.
멤버십 쿠팡 로켓와우, 네이버플러스 멤버십 60% 결제일 할인
할인 공통기준
지난달 1일~말일까지 LOCA LIKIT Eat 카드로 40만원 이상 이용 시 혜택이 제공됩니다.
할인한도는 모든 혜택 통합 월 1만 3천원 한도 내에서 제공합니다.
롯데카드 가맹점 및 업종 분류 기준으로 혜택을 제공합니다.
할인 적용 제외 대상 모든 무이자할부 이용금액, 포인트 충전, 기프트·선불카드 충전 및 구매, 상품권(모바일 상품권 포함) 구매''',
    },
]

# 표지·고지 중심 3페이지는 실제 카드 정보가 있는 2페이지로 교체합니다.
# 아래 전사본은 렌더링한 원본 PDF를 직접 읽어, 사람의 화면 읽기 순서로 작성했습니다.
PAGE_2_FULL_TEXT_GOLD = {
    ('samsung', 'Samsung_5_V4.pdf'): '''교육 5% 할인
학원·학습지·인터넷강의·서점 5% 결제일할인(청구할인)
업종 | 할인 대상점
학원 | 입시/보습·외국어·예체능계 학원
학습지 | 씽크빅, 교원, 대교, 한솔교육
인터넷강의 | 이투스, 메가스터디(엠베스트), 대성마이맥, 스카이에듀
서점 | 오프라인서점, 온라인서점(YES24, 인터파크 도서, 알라딘, 교보문고)
전월 이용금액대별 통합 월 할인한도
50만원 이상 | 100만원 이상 | 150만원 이상
7,000원 | 15,000원 | 30,000원
* 학원의 경우, 오프라인 일반 결제건에 한해 혜택이 제공됩니다.
* 인터넷강의의 경우, 공식 홈페이지를 통한 초·중·고 교육과정 결제건에 한해 혜택이 제공됩니다.
* 온라인서점의 경우, 공식 홈페이지·앱을 통한 결제건에 한해 혜택이 제공됩니다.
서비스 제공 기준
* 발급월+1개월까지는 전월 이용금액 50만원 미만 시에도 50만원 이상~100만원 미만 실적구간의 혜택이 제공됩니다. (전월 이용금액 100만원 이상 시에는 해당 실적구간 서비스 제공)
* 전월 이용금액이란, 매월 1일부터 말일까지 이용한 일시불 및 할부 이용금액을 의미하며, 단기카드대출(현금서비스), 장기카드대출(카드론), 각종 수수료 및 이자(할부수수료, 카드대출 이자 등), 연체료, 연회비 납부건은 일시불 및 할부 이용금액에 해당되지 않습니다.
* 전월 이용금액 산정 시, 건강보험/국민연금/고용보험/산재보험 및 장애인 고용부담금, 국세/지방세/공과금, 초·중·고등학교 학교납입금, 대학 등록금, 대중교통, 택시, 아파트 관리비, 부동산 임대료, 기프트/선불카드(포인트, 사이버머니 등 전자지급수단 포함) 구매 및 충전, 상품권 구매건은 제외됩니다.
* 삼성카드의 다른 결제일할인(청구할인) 혜택과 중복 적용되지 않으며, 할인 혜택이 큰 금액만 적용됩니다.
* 무이자할부, 삼성카드 할인이 적용된 일시불 및 할부 이용금액, 기프트/선불카드(포인트, 사이버머니 등 전자지급수단 포함) 구매 및 충전, 상품권 구매건은 할인에서 제외됩니다.
* 할인 혜택 및 전월 이용금액 산정은 승인 시점 기준으로 적용됩니다. 다만, 해외 결제건 및 무승인 결제건(예 : 대중교통, 이동통신 등 자동납부 결제건 등)의 경우 매출전표 접수 시점 기준으로 적용됩니다.
생활 필수 영역 2% 할인
할인점·온라인쇼핑몰·해외·의료·커피전문점·제과점 2% 결제일할인(청구할인)
업종 | 할인 대상점
할인점 | 이마트, 이마트 트레이더스, 롯데마트, 홈플러스
온라인쇼핑몰 | 삼성카드 쇼핑, G마켓, 옥션, 11번가, 인터파크, 쿠팡, 티몬, 위메프, SSG.COM
해외 | 해외 가맹점 및 해외 직접구매 이용건
의료 | 병·의원, 약국, 동물병원
커피전문점 | 스타벅스, 이디야커피, 투썸플레이스, 카페베네, 탐앤탐스, 커피빈, 엔제리너스, 할리스커피, 파스쿠찌, 아티제, 폴 바셋, 블루보틀
제과점 | 파리바게뜨, 뚜레쥬르, 던킨도너츠
전월 이용금액대별 통합 월 할인한도
50만원 이상 | 100만원 이상
7,000원 | 20,000원
* 할인점의 경우, 온라인몰도 포함되며, 기업형 슈퍼마켓(이마트 에브리데이, 롯데슈퍼, 홈플러스 익스프레스 등), 쇼핑 외 결제건(상품권, 주차장 등), 할인점 내 임대매장은 제외됩니다.
* 해외의 경우, 해외겸용카드에 한해 혜택이 제공됩니다.
* 해외 이용 시 별도의 수수료가 부과됩니다. 자세한 내용은 ‘유의사항’을 확인해 주세요.
* 의료의 경우, 오프라인 일반 결제건에 한하며, 요양병원, 보건소는 제외됩니다.
* 커피전문점·제과점의 경우, 오프라인 일반 결제건에 한하며, 백화점, 할인점, 쇼핑몰 내 임대매장은 제외됩니다.
* 스타벅스의 경우, 사이렌오더 결제건도 혜택이 제공됩니다.''',
    ('shinhan', 'Shinhan_Discount_Plan+_20250509.pdf'): '''신한카드 Discount Plan+
카드 디자인
연회비
구분 | 총 연회비 | 기본 연회비 | 서비스 연회비
국내전용 | Local | 5만원 | 7천원 | 4만3천원
해외겸용 | Mastercard Platinum | 5만원 | 7천원 | 4만3천원
※ 연회비는 기본연회비와 서비스연회비를 합산해 카드별로 청구됩니다.
※ 가족카드 연회비는 없습니다. (단, 가족카드 단독으로 발급받을 경우 연회비가 부과됩니다.)
※ 후불교통 기능이 있는 카드로만 신청 가능합니다.
※ 해외겸용(Mastercard)은 컨택리스 결제를 지원합니다.
※ 신한카드 Discount Plan+로 국제브랜드사(Mastercard)의 플래티늄 서비스를 이용할 수 있습니다.
브랜드사 서비스 안내장, 신한카드 홈페이지(www.shinhancard.com) 및 국제브랜드사 홈페이지(www.mastercard.co.kr)에서 서비스 세부 내용 및 유의사항을 확인해주세요.
한눈에 보기
Time Plan 서비스
· DAY 10% 할인 : 카페 / 음식점
· NIGHT 10% 할인 : 편의점 / 배달앱
Daily Plan 서비스
· 쇼핑 10% 할인 : 마트 / 온라인쇼핑 / 아울렛 / 잡화
· 이동 5% 할인 : 주유 / 카셰어링 / 주차 / 택시
· 생활 5% 할인 : 해외 / 병원·약국 / 미용실 / 온라인서점
Plan Day 서비스
· 매월 1일 Time Plan, Daily Plan 서비스 2배 할인(서비스당 1회)
Monthly Plan 서비스
· 정기결제 최대 20% 할인 - 공과금 10% / 디지털구독·멤버십 20% / 피트니스 5%
· 영화 예매 5천원 할인
· Monthly 리워드 최대 5천원 캐시백
Annual Plan 서비스
· 장보기 3만원 캐시백 (연 1회)
· 스피드메이트 정비 할인
· 테마파크 할인
[서비스별 월 통합 혜택 한도]
전월 이용금액(일시불+할부) | 40만원 이상 80만원 미만 | 80만원 이상 120만원 미만 | 120만원 이상 180만원 미만 | 180만원 이상
Time Plan | 5천원 | 1만원 | 1만5천원 | 2만원
Daily Plan | 1만 5천원 | 2만 5천원 | 3만5천원 | 5만원
Monthly Plan 정기결제 | 5천원 | 1만원 | 1만5천원 | 2만원
Monthly Plan 영화예매 | 5천원
Monthly Plan Monthly 리워드 | - | - | 3천원(150만원 이상 시) | 5천원
월 최대 혜택 한도 | 3만원 | 5만원 | 7만3천원* | 10만원
* 전월 이용금액 120만원 이상 150만원 미만일 경우 최대 7만원, 150만원 이상 180만원 미만일 경우 최대 7만3천원 혜택 한도가 제공됩니다.''',
    ('BC', 'BC_Business_Sky.pdf'): '''카드발급안내
상품기본정보
발행사 BC바로카드
브랜드 VISA
연회비 10,000원
등급 골드
구분 기업
발급여부 발급가능
연회비 반환 조건안내
• 카드 유효기간이 도래하기 전에 카드를 해지하는 경우 연회비 반환금액은 계약을 해지한 날로부터 일할 계산하여 산정하며, 10영업일 이내에 반환 처리됩니다.
• 다만, 부가서비스 제공내역 확인에 시간이 소요되는 등의 불가피한 사유로 10영업일 이내에 반환하기 어려운 경우 계약해지 날부터 3개월 이내에 반환할 수 있습니다.
• 이 경우 회원이 이미 납부한 연회비에 반영된 다음의 비용은 반환금액 산정에서 제외됩니다.
- 카드의 발행·배송 등 카드발급(신규발급)에 소요된 비용
- 카드 이용 시 제공되는 추가적인 혜택 등 부가서비스 제공에 소요된 비용''',
}
REPLACED_FULL_TEXT_KEYS = set(PAGE_2_FULL_TEXT_GOLD)
FULL_TEXT_GOLD_SAMPLES = [sample for sample in FULL_TEXT_GOLD_SAMPLES if (sample['issuer'], sample['file_name']) not in REPLACED_FULL_TEXT_KEYS]
FULL_TEXT_GOLD_SAMPLES.extend({
    'issuer': issuer, 'file_name': file_name, 'page_number': 2, 'gold_text': gold_text
} for (issuer, file_name), gold_text in PAGE_2_FULL_TEXT_GOLD.items())
FIELD_GOLD_SAMPLES = [
    {
        'issuer': 'lotte', 'file_name': 'Lotte_LOCA_LIKIT_Play.pdf', 'page_number': 3,
        'fields': {
            'fuel_discount_rate': '60%', 'fuel_brands': 'SK에너지, S-OIL, GS칼텍스, 현대오일뱅크',
            'movie_discount_rate': '60%', 'streaming_discount_rate': '60%',
            'minimum_spend': '40만원 이상', 'monthly_integrated_limit': '1만 3천원',
        },
    },
    {
        'issuer': 'NH', 'file_name': 'NH_AllWonderful.pdf', 'page_number': 1,
        'fields': {
            'local_total_annual_fee': '28,000원', 'local_basic_annual_fee': '6,000원',
            'local_affiliated_annual_fee': '22,000원', 'global_total_annual_fee': '30,000원',
            'global_basic_annual_fee': '6,000원', 'service_pack_change_frequency': '월 1회',
        },
    },
    {
        'issuer': 'hyundai', 'file_name': 'Hyundai_T_20260319.pdf', 'page_number': 5,
        'fields': {
            'international_brand_fee_rate': '1.1%', 'overseas_service_fee_rate': '0.2%',
            'fee_discount_rate': '100%', 'metal_plate_fee': '7만원',
            'metal_plate_brand': 'Visa Platinum', 'metal_plate_delivery_period': '평균 3주 내외',
        },
    },
    {
        'issuer': 'hana', 'file_name': 'Hana_WonderCard2.0.pdf', 'page_number': 12,
        'fields': {
            'monthly_cashback': '1만원', 'annual_cashback': '1만원',
            'monthly_spend_requirement': '40만원 이상', 'continuous_use_requirement': '6개월 연속 1원 이상',
            'benefit_provision_timing': '다음달 10영업일 이내', 'maximum_family_cards': '4장',
        },
    },
    {
        'issuer': 'kookmin', 'file_name': 'Kookmin_My_WE_SH_20250102.pdf', 'page_number': 3,
        'fields': {
            'delivery_coffee_discount_rate': '5%', 'movie_discount_rate': '30%',
            'delivery_coffee_monthly_limit': '5천원', 'management_monthly_limit': '1만원',
            'minimum_spend': '40만원 이상', 'movie_annual_limit': '연 4회',
        },
    },
    {
        'issuer': 'samsung', 'file_name': 'Samsung_iD_ALL.pdf', 'page_number': 2,
        'fields': {
            'automatic_discount_rate': '5%', 'minimum_spend_tier': '40만원 이상',
            'first_tier_monthly_limit': '5,000원', 'second_tier_spend': '70만원 이상',
            'second_tier_monthly_limit': '10,000원', 'base_discount_rate': '0.5%',
        },
    },
    {
        'issuer': 'lotte', 'file_name': 'Lotte_LOCA_LIKIT_Eat.pdf', 'page_number': 3,
        'fields': {
            'restaurant_discount_rate': '60%', 'delivery_app_discount_rate': '60%',
            'coffee_discount_rate': '60%', 'membership_discount_rate': '60%',
            'minimum_spend': '40만원 이상', 'monthly_integrated_limit': '1만 3천원',
        },
    },
    {
        'issuer': 'samsung', 'file_name': 'Samsung_5_V4.pdf', 'page_number': 2,
        'fields': {
            'education_discount_rate': '5%', 'education_50m_limit': '7,000원',
            'education_100m_limit': '15,000원', 'education_150m_limit': '30,000원',
            'essential_discount_rate': '2%', 'essential_100m_limit': '20,000원',
        },
    },
    {
        'issuer': 'shinhan', 'file_name': 'Shinhan_Discount_Plan+_20250509.pdf', 'page_number': 2,
        'fields': {
            'local_annual_fee': '5만원', 'mastercard_annual_fee': '5만원',
            'time_plan_discount': '10%', 'daily_plan_discount': '10%',
            'daily_plan_mobility_discount': '5%', 'monthly_movie_discount': '5천원',
        },
    },
    {
        'issuer': 'BC', 'file_name': 'BC_Business_Sky.pdf', 'page_number': 2,
        'fields': {
            'issuer_name': 'BC바로카드', 'brand': 'VISA', 'annual_fee': '10,000원',
            'grade': '골드', 'card_type': '기업', 'issuance_status': '발급가능',
        },
    },
]

CRITICAL_FACTS = {
    ('samsung', 'Samsung_5_V4.pdf', 2): ['학원·학습지·인터넷강의·서점 5% 결제일할인', '7,000원 | 15,000원 | 30,000원', '생활 필수 영역 2% 할인'],
    ('lotte', 'Lotte_LOCA_LIKIT_Play.pdf', 3): ['SK에너지, S-OIL, GS칼텍스, 현대오일뱅크', '60% 결제일 할인', '모든 혜택 통합 월 1만 3천원'],
    ('shinhan', 'Shinhan_Discount_Plan+_20250509.pdf', 2): ['국내전용 | Local | 5만원', '이동 5% 할인 : 주유 / 카셰어링 / 주차 / 택시', '월 최대 혜택 한도 | 3만원 | 5만원 | 7만3천원'],
    ('NH', 'NH_AllWonderful.pdf', 1): ['국내전용 Local 28,000원', '국내외겸용 Mastercard UnionPay 30,000원', '월 1회 변경가능'],
    ('hyundai', 'Hyundai_T_20260319.pdf', 5): ['국제브랜드 수수료(Visa) : 1.1%', '해외서비스 수수료 : 0.2%', '발급 수수료 7만원'],
    ('BC', 'BC_Business_Sky.pdf', 2): ['발행사 BC바로카드', '연회비 10,000원', '10영업일 이내에 반환 처리됩니다'],
    ('hana', 'Hana_WonderCard2.0.pdf', 12): ['가족카드당 1만원 캐시백(월 1회)', '직전 6개월 연속 1원 이상 이용', '가족카드 최대 4장'],
    ('kookmin', 'Kookmin_My_WE_SH_20250102.pdf', 3): ['배달/커피 5% 할인', '영화관 30% 할인', '전월 이용실적 40만원 이상'],
    ('samsung', 'Samsung_iD_ALL.pdf', 2): ['1개 영역에 대해 5% 결제일할인', '40만원 이상 5,000원', '70만원 이상 10,000원'],
    ('lotte', 'Lotte_LOCA_LIKIT_Eat.pdf', 3): ['음식점 60% 결제일 할인', '배달의 민족, 쿠팡이츠, 요기요 60% 결제일 할인', '모든 혜택 통합 월 1만 3천원'],
}
for sample in FULL_TEXT_GOLD_SAMPLES:
    sample['critical_facts'] = CRITICAL_FACTS[(sample['issuer'], sample['file_name'], sample['page_number'])]

GROUND_TRUTH_PATH = PROJECT_ROOT / 'notebooks' / 'data' / '02_pdf_text_layer_check' / 'ocr_evaluation' / '02_pymupdf_vision_full_text_gold.json'
ground_truth_payload = {
    'schema_version': '1.0',
    'purpose': 'PyMuPDF와 Vision OCR의 페이지 전체 텍스트 추출 성능 비교',
    'full_text_case_schema': {'issuer': 'str', 'file_name': 'str', 'page_number': 'int', 'gold_text': 'str', 'critical_facts': 'list[str]'},
    'field_case_schema': {'issuer': 'str', 'file_name': 'str', 'page_number': 'int', 'fields': 'dict[str, str]'},
    'label_source': '원본 PDF 페이지를 시각적으로 확인한 전체 텍스트 전사',
    'label_scope': '사람이 읽을 수 있는 텍스트. 장식 이미지의 위치와 시각 디자인은 포함하지 않음.',
    'normalization_policy': 'OCR의 Markdown 표기, 글머리표, 제어문자는 내용 오류와 분리해 정규화함.',
    'full_text_cases': FULL_TEXT_GOLD_SAMPLES,
    'field_cases': FIELD_GOLD_SAMPLES,
}
GROUND_TRUTH_PATH.parent.mkdir(parents=True, exist_ok=True)
GROUND_TRUTH_PATH.write_text(json.dumps(ground_truth_payload, ensure_ascii=False, indent=2), encoding='utf-8')
saved_ground_truth = json.loads(GROUND_TRUTH_PATH.read_text(encoding='utf-8'))
assert saved_ground_truth['full_text_cases'] == FULL_TEXT_GOLD_SAMPLES, '전체 텍스트 정답셋 저장 후 내용 불일치'
assert saved_ground_truth['field_cases'] == FIELD_GOLD_SAMPLES, '필드 정답셋 저장 후 내용 불일치'
print(f'정답 페이지 수: {len(FULL_TEXT_GOLD_SAMPLES) + len(FIELD_GOLD_SAMPLES)}')
print(f'정답셋 저장 경로: {GROUND_TRUTH_PATH}')

def normalize_for_ocr_metric(text: str) -> str:
    # OCR 결과의 Markdown 표·줄바꿈 표기는 내용 오류로 계산하지 않습니다.
    text = unicodedata.normalize('NFKC', text)
    text = re.sub(r'<br\s*/?>', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'(?m)^\s*\|[\s:|-]+\|\s*$', ' ', text)
    text = re.sub(r'[|•ㆍ·★※]', ' ', text)
    text = re.sub(r'[\x00-\x1f\x7f-\x9f]', '', text)
    return re.sub(r'\s+', '', text)

def word_tokens_for_metric(text: str) -> list[str]:
    return re.findall(r'[가-힣A-Za-z]+|\d+(?:[,.]\d+)?%?', unicodedata.normalize('NFKC', text))

def levenshtein_distance(reference, hypothesis) -> int:
    if len(reference) < len(hypothesis):
        reference, hypothesis = hypothesis, reference
    previous = list(range(len(hypothesis) + 1))
    for reference_index, reference_value in enumerate(reference, start=1):
        current = [reference_index]
        for hypothesis_index, hypothesis_value in enumerate(hypothesis, start=1):
            current.append(min(
                current[-1] + 1,
                previous[hypothesis_index] + 1,
                previous[hypothesis_index - 1] + (reference_value != hypothesis_value),
            ))
        previous = current
    return previous[-1]

def error_rate(reference, hypothesis) -> float:
    return levenshtein_distance(reference, hypothesis) / len(reference) if reference else 0.0


정답 페이지 수: 20
정답셋 저장 경로: notebooks/data/02_pdf_text_layer_check/ocr_evaluation/02_pymupdf_vision_full_text_gold.json


In [12]:
evaluation_rows = []

for sample in FULL_TEXT_GOLD_SAMPLES:
    pdf_path = RAW_PDF_DIR / sample['issuer'] / sample['file_name']
    vision_path = VISION_DIR / sample['issuer'] / f"{Path(sample['file_name']).stem}.txt"
    with fitz.open(pdf_path) as document:
        pymupdf_text = document[sample['page_number'] - 1].get_text('text')
    vision_text = read_vision_pages(vision_path).get(sample['page_number'], '')

    gold_chars = normalize_for_ocr_metric(sample['gold_text'])
    gold_words = word_tokens_for_metric(sample['gold_text'])
    for engine, output_text in [('PyMuPDF', pymupdf_text), ('Vision OCR', vision_text)]:
        output_chars = normalize_for_ocr_metric(output_text)
        output_words = word_tokens_for_metric(output_text)
        evaluation_rows.append({
            'issuer': sample['issuer'],
            'file_name': sample['file_name'],
            'page_number': sample['page_number'],
            'engine': engine,
            'gold_chars': len(gold_chars),
            'output_chars': len(output_chars),
            'CER': error_rate(gold_chars, output_chars),
            'gold_words': len(gold_words),
            'output_words': len(output_words),
            'WER': error_rate(gold_words, output_words),
        })

        output_token_set = set(output_words)
        fact_token_coverages = []
        for fact in sample['critical_facts']:
            fact_tokens = word_tokens_for_metric(fact)
            coverage = sum(token in output_token_set for token in fact_tokens) / len(fact_tokens) if fact_tokens else 0.0
            fact_token_coverages.append(coverage)
        fact_hits = sum(
            normalize_for_ocr_metric(fact) in output_chars
            for fact in sample['critical_facts']
        )
        evaluation_rows[-1]['critical_fact_substring_match'] = fact_hits / len(sample['critical_facts'])
        evaluation_rows[-1]['critical_fact_token_coverage'] = sum(fact_token_coverages) / len(fact_token_coverages)

ocr_metric_df = pd.DataFrame(evaluation_rows)
display(ocr_metric_df.style.format({'CER': '{:.2%}', 'WER': '{:.2%}'}))

ocr_metric_summary = (
    ocr_metric_df.groupby('engine', as_index=False)
    .agg(
        pages=('page_number', 'count'),
        mean_CER=('CER', 'mean'),
        mean_WER=('WER', 'mean'),
        total_gold_chars=('gold_chars', 'sum'),
        total_output_chars=('output_chars', 'sum'),
        critical_fact_substring_match=('critical_fact_substring_match', 'mean'),
        critical_fact_token_coverage=('critical_fact_token_coverage', 'mean'),
    )
)
display(ocr_metric_summary.style.format({'mean_CER': '{:.2%}', 'mean_WER': '{:.2%}', 'critical_fact_substring_match': '{:.2%}', 'critical_fact_token_coverage': '{:.2%}'}))

print('해석 주의: CER/WER는 원본을 읽는 순서와 누락까지 함께 반영합니다.')

field_metric_rows = []
for sample in FIELD_GOLD_SAMPLES:
    pdf_path = RAW_PDF_DIR / sample['issuer'] / sample['file_name']
    vision_path = VISION_DIR / sample['issuer'] / f"{Path(sample['file_name']).stem}.txt"
    with fitz.open(pdf_path) as document:
        engine_outputs = {
            'PyMuPDF': document[sample['page_number'] - 1].get_text('text'),
            'Vision OCR': read_vision_pages(vision_path).get(sample['page_number'], ''),
        }
    for engine, output_text in engine_outputs.items():
        normalized_output = normalize_for_ocr_metric(output_text)
        value_hits = sum(
            normalize_for_ocr_metric(value) in normalized_output
            for value in sample['fields'].values()
        )
        field_metric_rows.append({
            'issuer': sample['issuer'],
            'file_name': sample['file_name'],
            'page_number': sample['page_number'],
            'engine': engine,
            'field_count': len(sample['fields']),
            'field_value_recall': value_hits / len(sample['fields']),
        })

field_metric_df = pd.DataFrame(field_metric_rows)
field_metric_summary = field_metric_df.groupby('engine', as_index=False).agg(
    pages=('page_number', 'count'),
    field_value_recall=('field_value_recall', 'mean'),
)
display(field_metric_df.style.format({'field_value_recall': '{:.2%}'}))
display(field_metric_summary.style.format({'field_value_recall': '{:.2%}'}))

# 동일한 10개 페이지의 전체 텍스트·필드 평가를 카드/페이지/엔진 단위로 결합합니다.
page_metric_df = (
    ocr_metric_df.merge(
        field_metric_df,
        on=['issuer', 'file_name', 'page_number', 'engine'],
        how='inner',
    )
    .sort_values(['issuer', 'file_name', 'page_number', 'engine'])
    .reset_index(drop=True)
)
assert len(page_metric_df) == len(FULL_TEXT_GOLD_SAMPLES) * 2, '페이지별 평가 결과 수가 10페이지 x 2엔진과 다릅니다.'
display(page_metric_df[[
    'issuer', 'file_name', 'page_number', 'engine', 'CER', 'WER',
    'critical_fact_substring_match', 'critical_fact_token_coverage',
    'field_count', 'field_value_recall',
]].style.format({
    'CER': '{:.2%}', 'WER': '{:.2%}',
    'critical_fact_substring_match': '{:.2%}',
    'critical_fact_token_coverage': '{:.2%}',
    'field_value_recall': '{:.2%}',
}))

print(f'동일한 {len(FULL_TEXT_GOLD_SAMPLES)}페이지에 대해 전체 텍스트와 필드 값 평가를 모두 실행했습니다.')
print('이 결과는 원본 PDF 기반 표본 평가이며, 전체 문서군의 일반 성능을 뜻하지 않습니다.')

,issuer,file_name,page_number,engine,gold_chars,output_chars,CER,gold_words,output_words,WER,critical_fact_substring_match,critical_fact_token_coverage
0,lotte,Lotte_LOCA_LIKIT_Play.pdf,3,PyMuPDF,475,482,52.00%,158,158,51.90%,1.000000,1.000000
1,lotte,Lotte_LOCA_LIKIT_Play.pdf,3,Vision OCR,475,475,0.00%,158,158,0.00%,1.000000,1.000000
2,NH,NH_AllWonderful.pdf,1,PyMuPDF,616,601,22.56%,183,179,26.23%,1.000000,1.000000
3,NH,NH_AllWonderful.pdf,1,Vision OCR,616,621,0.81%,183,185,1.64%,1.000000,1.000000
4,hyundai,Hyundai_T_20260319.pdf,5,PyMuPDF,450,456,80.22%,162,164,83.95%,1.000000,1.000000
5,hyundai,Hyundai_T_20260319.pdf,5,Vision OCR,450,452,0.44%,162,162,0.00%,1.000000,1.000000
6,hana,Hana_WonderCard2.0.pdf,12,PyMuPDF,568,572,1.41%,223,225,2.69%,1.000000,0.916667
7,hana,Hana_WonderCard2.0.pdf,12,Vision OCR,568,568,0.70%,223,223,0.00%,1.000000,0.916667
8,kookmin,Kookmin_My_WE_SH_20250102.pdf,3,PyMuPDF,537,584,25.88%,177,197,36.72%,1.000000,1.000000
9,kookmin,Kookmin_My_WE_SH_20250102.pdf,3,Vision OCR,537,584,12.10%,177,198,16.95%,1.000000,1.000000


,engine,pages,mean_CER,mean_WER,total_gold_chars,total_output_chars,critical_fact_substring_match,critical_fact_token_coverage
0,PyMuPDF,10,28.20%,30.60%,6161,6202,80.00%,98.69%
1,Vision OCR,10,6.33%,6.52%,6161,6203,93.33%,99.17%


해석 주의: CER/WER는 원본을 읽는 순서와 누락까지 함께 반영합니다.


,issuer,file_name,page_number,engine,field_count,field_value_recall
0,lotte,Lotte_LOCA_LIKIT_Play.pdf,3,PyMuPDF,6,100.00%
1,lotte,Lotte_LOCA_LIKIT_Play.pdf,3,Vision OCR,6,100.00%
2,NH,NH_AllWonderful.pdf,1,PyMuPDF,6,100.00%
3,NH,NH_AllWonderful.pdf,1,Vision OCR,6,100.00%
4,hyundai,Hyundai_T_20260319.pdf,5,PyMuPDF,6,100.00%
5,hyundai,Hyundai_T_20260319.pdf,5,Vision OCR,6,100.00%
6,hana,Hana_WonderCard2.0.pdf,12,PyMuPDF,6,100.00%
7,hana,Hana_WonderCard2.0.pdf,12,Vision OCR,6,100.00%
8,kookmin,Kookmin_My_WE_SH_20250102.pdf,3,PyMuPDF,6,100.00%
9,kookmin,Kookmin_My_WE_SH_20250102.pdf,3,Vision OCR,6,100.00%


,engine,pages,field_value_recall
0,PyMuPDF,10,100.00%
1,Vision OCR,10,100.00%


,issuer,file_name,page_number,engine,CER,WER,critical_fact_substring_match,critical_fact_token_coverage,field_count,field_value_recall
0,BC,BC_Business_Sky.pdf,2,PyMuPDF,9.56%,10.75%,33.33%,100.00%,6,100.00%
1,BC,BC_Business_Sky.pdf,2,Vision OCR,0.00%,0.00%,100.00%,100.00%,6,100.00%
2,NH,NH_AllWonderful.pdf,1,PyMuPDF,22.56%,26.23%,100.00%,100.00%,6,100.00%
3,NH,NH_AllWonderful.pdf,1,Vision OCR,0.81%,1.64%,100.00%,100.00%,6,100.00%
4,hana,Hana_WonderCard2.0.pdf,12,PyMuPDF,1.41%,2.69%,100.00%,91.67%,6,100.00%
5,hana,Hana_WonderCard2.0.pdf,12,Vision OCR,0.70%,0.00%,100.00%,91.67%,6,100.00%
6,hyundai,Hyundai_T_20260319.pdf,5,PyMuPDF,80.22%,83.95%,100.00%,100.00%,6,100.00%
7,hyundai,Hyundai_T_20260319.pdf,5,Vision OCR,0.44%,0.00%,100.00%,100.00%,6,100.00%
8,kookmin,Kookmin_My_WE_SH_20250102.pdf,3,PyMuPDF,25.88%,36.72%,100.00%,100.00%,6,100.00%
9,kookmin,Kookmin_My_WE_SH_20250102.pdf,3,Vision OCR,12.10%,16.95%,100.00%,100.00%,6,100.00%


동일한 10페이지에 대해 전체 텍스트와 필드 값 평가를 모두 실행했습니다.
이 결과는 원본 PDF 기반 표본 평가이며, 전체 문서군의 일반 성능을 뜻하지 않습니다.


## 오류 유형 분류 (10페이지 보조 분석)

CER/WER만으로는 오류 원인을 구분할 수 없습니다. 아래 셀은 단어 멀티셋 기준의 누락·삽입과 단어 순서 유사도를 보조 지표로 계산하고, 원본 PDF를 다시 확인한 수동 판정을 함께 표시합니다.

- 문자/숫자 오류: 원문 단어가 다른 단어로 바뀐 경우
- 누락·삽입: 원문에 있는 단어가 사라지거나 원문에 없는 단어가 추가된 경우
- 읽기 순서: 동일한 내용이지만 페이지 좌표·객체 저장 순서 때문에 배열이 달라진 경우
- 레이아웃: 표의 행·열 관계, 제목과 본문의 연결처럼 단순 텍스트 비교만으로 판단하기 어려운 경우

In [13]:
from difflib import SequenceMatcher

def counter_items(counter: Counter, limit: int = 8) -> str:
    items = []
    for token, count in counter.most_common(limit):
        items.append(f'{token}({count})' if count > 1 else token)
    return ', '.join(items) or '-'

diagnostic_rows = []
for sample in FULL_TEXT_GOLD_SAMPLES:
    pdf_path = RAW_PDF_DIR / sample['issuer'] / sample['file_name']
    vision_path = VISION_DIR / sample['issuer'] / f"{Path(sample['file_name']).stem}.txt"
    with fitz.open(pdf_path) as document:
        engine_outputs = {
            'PyMuPDF': document[sample['page_number'] - 1].get_text('text'),
            'Vision OCR': read_vision_pages(vision_path).get(sample['page_number'], ''),
        }
    gold_words = word_tokens_for_metric(sample['gold_text'])
    gold_counter = Counter(gold_words)
    for engine, output_text in engine_outputs.items():
        output_words = word_tokens_for_metric(output_text)
        output_counter = Counter(output_words)
        common_count = sum((gold_counter & output_counter).values())
        missing = gold_counter - output_counter
        extra = output_counter - gold_counter
        diagnostic_rows.append({
            'issuer': sample['issuer'],
            'file_name': sample['file_name'],
            'engine': engine,
            'content_recall': common_count / len(gold_words),
            'content_precision': common_count / len(output_words),
            'word_sequence_similarity': SequenceMatcher(None, gold_words, output_words).ratio(),
            'missing_words': counter_items(missing),
            'extra_words': counter_items(extra),
        })

diagnostic_df = pd.DataFrame(diagnostic_rows)
display(diagnostic_df.style.format({
    'content_recall': '{:.2%}',
    'content_precision': '{:.2%}',
    'word_sequence_similarity': '{:.2%}',
}))

diagnostic_summary = (
    diagnostic_df.groupby('engine', as_index=False)
    .agg(
        pages=('file_name', 'count'),
        mean_content_recall=('content_recall', 'mean'),
        mean_content_precision=('content_precision', 'mean'),
        mean_word_sequence_similarity=('word_sequence_similarity', 'mean'),
    )
)
display(diagnostic_summary.style.format({
    'mean_content_recall': '{:.2%}',
    'mean_content_precision': '{:.2%}',
    'mean_word_sequence_similarity': '{:.2%}',
}))

# 원본 PDF 렌더링과 두 출력문을 대조해 작성한 초기 수동 판정입니다.
manual_error_classification = pd.DataFrame([
    {
        'issuer': 'samsung',
        'engine': 'PyMuPDF',
        'primary_error_type': '읽기 순서 + 누락',
        'original_pdf_check': '본문은 대체로 읽히지만 대표 문구(FOR EARTH FOR US), 환경 문구, 609, Samsung Card 표기가 빠졌고, 화면 순서와 다른 순서로 출력됩니다.',
    },
    {
        'issuer': 'samsung',
        'engine': 'Vision OCR',
        'primary_error_type': '확인된 오류 없음',
        'original_pdf_check': '이 초기 표본의 사람이 읽는 텍스트 기준으로 누락·오독을 확인하지 못했습니다. 로고의 시각적 위치 자체는 평가하지 않았습니다.',
    },
    {
        'issuer': 'lotte',
        'engine': 'PyMuPDF',
        'primary_error_type': '읽기 순서',
        'original_pdf_check': '혜택 문구와 수치는 대부분 존재하지만 제목과 주유·영화·스트리밍·멤버십 블록의 출력 순서가 화면 읽기 순서와 다릅니다.',
    },
    {
        'issuer': 'lotte',
        'engine': 'Vision OCR',
        'primary_error_type': '확인된 오류 없음',
        'original_pdf_check': '이 초기 표본의 사람이 읽는 텍스트 기준으로 누락·오독을 확인하지 못했습니다.',
    },
    {
        'issuer': 'shinhan',
        'engine': 'PyMuPDF',
        'primary_error_type': '읽기 순서 + 문자/단어 오류',
        'original_pdf_check': '고객센터 라벨과 전화번호, 제목과 약관 문구가 존재하지만 화면의 좌상단·우상단·하단 블록을 사람이 읽는 순서로 복원하지 못했습니다. 또한 원문의 과도한을 과도할로 출력한 1건을 확인했습니다.',
    },
    {
        'issuer': 'shinhan',
        'engine': 'Vision OCR',
        'primary_error_type': '문자/단어 오류',
        'original_pdf_check': '원문의 이용대금을 이용금액으로, 과도한을 과도할로 출력한 각 1건을 확인했습니다. 비은행 금융기관을 비은행금융기관으로 출력한 차이는 띄어쓰기 결합입니다.',
    },
])
display(manual_error_classification)

,issuer,file_name,engine,content_recall,content_precision,word_sequence_similarity,missing_words,extra_words
0,lotte,Lotte_LOCA_LIKIT_Play.pdf,PyMuPDF,100.00%,100.00%,68.99%,-,-
1,lotte,Lotte_LOCA_LIKIT_Play.pdf,Vision OCR,100.00%,100.00%,100.00%,-,-
2,NH,NH_AllWonderful.pdf,PyMuPDF,96.72%,98.88%,86.74%,"NH올원더풀, 모든, 순간, 원더풀하게, 채워지다, PACK과","PACK, 과"
3,NH,NH_AllWonderful.pdf,Vision OCR,99.45%,98.38%,98.91%,PACK과,"br, PACK, 과"
4,hyundai,Hyundai_T_20260319.pdf,PyMuPDF,100.00%,98.78%,52.15%,-,"02, 03"
5,hyundai,Hyundai_T_20260319.pdf,Vision OCR,100.00%,100.00%,100.00%,-,-
6,hana,Hana_WonderCard2.0.pdf,PyMuPDF,98.21%,97.33%,97.77%,"본인가드(2), 본인가드와, 본인가드를","본인카드(2), 20, 21, 본인카드와, 본인카드를"
7,hana,Hana_WonderCard2.0.pdf,Vision OCR,100.00%,100.00%,100.00%,-,-
8,kookmin,Kookmin_My_WE_SH_20250102.pdf,PyMuPDF,96.61%,86.80%,80.21%,"진심, 배달업, 노는데, 5, 천원, 로부터","제외(3), 매장(3), 등(2), 백화점(2), 대형마트(2), 일부(2), 입점(2), 배달앱"
9,kookmin,Kookmin_My_WE_SH_20250102.pdf,Vision OCR,97.18%,86.87%,91.73%,"진심, 노는데, 5, 천원, 로부터","제외(3), 매장(3), 등(2), 백화점(2), 대형마트(2), 일부(2), 입점(2), 오프라인"


,engine,pages,mean_content_recall,mean_content_precision,mean_word_sequence_similarity
0,PyMuPDF,10,99.00%,98.15%,80.45%
1,Vision OCR,10,99.43%,98.42%,96.70%


,issuer,engine,primary_error_type,original_pdf_check
0,samsung,PyMuPDF,읽기 순서 + 누락,"본문은 대체로 읽히지만 대표 문구(FOR EARTH FOR US), 환경 문구, 6..."
1,samsung,Vision OCR,확인된 오류 없음,이 초기 표본의 사람이 읽는 텍스트 기준으로 누락·오독을 확인하지 못했습니다. 로고...
2,lotte,PyMuPDF,읽기 순서,혜택 문구와 수치는 대부분 존재하지만 제목과 주유·영화·스트리밍·멤버십 블록의 출력...
3,lotte,Vision OCR,확인된 오류 없음,이 초기 표본의 사람이 읽는 텍스트 기준으로 누락·오독을 확인하지 못했습니다.
4,shinhan,PyMuPDF,읽기 순서 + 문자/단어 오류,"고객센터 라벨과 전화번호, 제목과 약관 문구가 존재하지만 화면의 좌상단·우상단·하단..."
5,shinhan,Vision OCR,문자/단어 오류,"원문의 이용대금을 이용금액으로, 과도한을 과도할로 출력한 각 1건을 확인했습니다. ..."
